# Spatial block cross-validation

This short demo shows how to compare spatial regression models with `bayespecon.diagnostics.spatial_kfold`, a refit-based spatial k-fold predictive evaluator following [Roberts *et al.* (2017)]().

For each fold, the model is refit on the training observations and the held-out fold's log predictive density is evaluated under the
**full-data joint Gaussian** implied by the model:

$$
\log p(y_{\text{test}} \mid y_{\text{train}}, \theta)
  = \tfrac{1}{2}\log|\Lambda_{tt}|
  - \tfrac{n_{\text{test}}}{2}\log(2\pi)
  - \tfrac{1}{2}\, z_{\text{test}}^{\top}\,\Lambda_{tt}^{-1}\, z_{\text{test}},
$$

with $\Lambda = A^{\top}A/\sigma^{2}$ where $A = I - \rho W$
(SAR / SDM), $A = I - \lambda W$ (SEM / SDEM), or $A = I$ (OLS / SLX),
and $z = \Lambda(y - \mu)$. This avoids the well-known failure modes of
PSIS-LOO on spatially dependent data.


## Known-truth setup

We simulate from a SAR data-generating process on a regular grid so the "correct" model is known in advance. A well-behaved CV criterion should prefer the SAR over the misspecified OLS, SLX, and SEM alternatives.


In [ ]:
import numpy as np
import pandas as pd
from libpysal.graph import Graph

from bayespecon.dgp import simulate_sar
from bayespecon.models import OLS, SAR, SEM, SLX
from bayespecon.diagnostics import spatial_kfold

SEED = 0
GRID = 12  # 12 x 12 = 144 cells
RHO_TRUE = 0.6
BETA_TRUE = np.array([1.0, 2.0, -1.5])

gdf = simulate_sar(
    n=GRID,
    rho=RHO_TRUE,
    beta=BETA_TRUE,
    sigma=1.0,
    seed=SEED,
    create_gdf=True,
    geometry_type="polygon",
)

W = Graph.build_contiguity(gdf, rook=True).transform("r")
print(f"n = {len(gdf)}, true rho = {RHO_TRUE}, columns = {list(gdf.columns)}")


## Fit candidate models

We fit four candidates — only the SAR matches the DGP — using short
MCMC runs sufficient for a pedagogical example.


In [ ]:
FIT_KW = dict(draws=400, tune=400, chains=2, random_seed=SEED, progressbar=False)
FORMULA = "y ~ X_1 + X_2"

models = {
    "OLS": OLS(formula=FORMULA, data=gdf, W=W),
    "SLX": SLX(formula=FORMULA, data=gdf, W=W),
    "SAR": SAR(formula=FORMULA, data=gdf, W=W, logdet_method="eigenvalue"),
    "SEM": SEM(formula=FORMULA, data=gdf, W=W, logdet_method="eigenvalue"),
}
for name, m in models.items():
    m.fit(**FIT_KW)
    print(f"  fit {name}")


## Spatial block CV

`spatial_kfold` derives 5 spatial blocks from the cell centroids via
KMeans, refits each model on each train fold, and accumulates the
held-out elpd. Higher elpd is better.


In [ ]:
CV_KW = dict(draws=200, tune=200, chains=1, random_seed=SEED, progressbar=False)

results = {}
for name, m in models.items():
    results[name] = spatial_kfold(
        m, geometry=gdf.geometry, n_blocks=5, **CV_KW
    )

summary = pd.DataFrame(
    {
        "elpd": [r.elpd for r in results.values()],
        "se": [r.se for r in results.values()],
        "n_folds": [r.n_folds for r in results.values()],
    },
    index=list(results.keys()),
).sort_values("elpd", ascending=False)
summary["delta_elpd"] = summary["elpd"] - summary["elpd"].max()
summary


With data simulated from a SAR DGP, spatial block CV should rank `SAR`
on top, with the misspecified alternatives trailing behind. The
`delta_elpd` column shows the elpd difference relative to the best
model — values close to zero indicate near-equivalent predictive
performance, while large negative values indicate worse out-of-block
prediction.
